# Full Pipeline — Playground

End-to-end run of the **entire stock-selection pipeline** in one notebook, top to bottom.
Each stage below has a short explainer of *what* it does and *which dials* (in
`backend/config.py`) govern it.

```
Stage 1  Universe Filter   weekly   ~60-80 large-cap liquid stocks  → watchlist.csv
Stage 2  Momentum Scanner  nightly  scores the watchlist 0-3        → top candidates
Gate 1   Hard Threat       rules    blocks on macro/market shocks   (no Claude)
Gate 2   News Threat       Claude   blocks on catastrophic news
Gate 3   Sentiment         Claude   direction + confidence          → pass/block/caution
Gate 4   Contradiction     Claude   stock-vs-market divergence
Gate 5   Edge / EV         rules    win-probability + expected value → BUY / SKIP
Risk     Risk Gate         rules    sizing + portfolio limits         → approved / rejected
```

The funnel narrows at every step: a few dozen universe names → a handful of momentum
candidates → the gates filter those down → the **Risk Gate** sizes and approves the final
**BUY** list.

> **⚠️ Live API + Claude calls.** Gates 2-4 each make one Claude (Haiku) call, so
> ~3 calls per surviving candidate. Needs `.env` keys: `ANTHROPIC_API_KEY` plus a news source
> (`ALPACA_API_KEY` + `ALPACA_SECRET_KEY`, or `NEWS_API_KEY`); the earnings/universe steps use
> `FINNHUB_API_KEY`. Missing keys degrade gracefully — gates **block** rather than crash.

> **Every tunable number lives in `backend/config.py`** — the single dial board. This notebook
> only *reads* those values; tweak the strategy there.

## Setup — imports & path wiring

The numbered folders (`01_scanner`, `02_intelligence`, `03_risk`) are **not** importable Python packages,
so — exactly like every other playground — we add the relevant directories to `sys.path` and
import each module by its bare name. We also add `backend/` itself so `from config import ...`
works here too.

Both `universe_filter` and `momentum_scanner` define a **cwd-relative** `WATCHLIST_PATH`
(`'data/watchlist.csv'`). We re-point both at the absolute path so the notebook runs no matter
where the kernel was launched.

In [1]:
import sys
import pathlib
import pandas as pd

# --- Anchor to backend/ regardless of where the kernel started -------------
nb_dir = pathlib.Path('.').resolve()
if (nb_dir / 'config.py').exists():
    backend_dir = nb_dir                                  # launched from backend/
else:
    backend_dir = pathlib.Path('backend').resolve()       # launched from repo root

intelligence_dir = backend_dir / '02_intelligence'
scanner_dir      = backend_dir / '01_scanner'
risk_dir         = backend_dir / '03_risk'

# Bare-name imports need each module's own dir on sys.path.
for p in [
    backend_dir,                                  # → config.py
    scanner_dir,                                  # → universe_filter, momentum_scanner
    intelligence_dir,                             # → constants, helpers/*
    risk_dir,                                     # → risk_gate
    intelligence_dir / 'gate1_hard_threat',
    intelligence_dir / 'gate2_news_threat',
    intelligence_dir / 'gate3_sentiment',
    intelligence_dir / 'gate4_contradiction',
    intelligence_dir / 'gate5_signal',
]:
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

# --- Stage 1 & 2: scanner --------------------------------------------------
import universe_filter
import momentum_scanner
from universe_filter import run_universe_filter
from momentum_scanner import run_scan

# Re-point both cwd-relative watchlist paths at the real file.
WATCHLIST_PATH = str(scanner_dir / 'data' / 'watchlist.csv')
universe_filter.WATCHLIST_PATH = WATCHLIST_PATH
momentum_scanner.WATCHLIST_PATH = WATCHLIST_PATH

# --- Gates 1-5 -------------------------------------------------------------
from hard_threat_gate1 import get_shared_market_data, screen_gate1_hard_threats
from news_threat_gate2 import assess_gate2_news_threat
from sentiment_gate3 import evaluate_gate3_sentiment
from contradiction_gate4 import detect_gate4_contradiction
from signal_gate5 import decide_gate5_signal
from risk_gate import validate_trade

# --- Fetchers shared across gates ------------------------------------------
from helpers.fetchers.news import fetch_news
from helpers.fetchers.market import get_market_context

# --- Active dials (read-only echo) -----------------------------------------
import config
print('Active strategy dials (edit in backend/config.py):')
print(f"  Universe : MIN_MARKET_CAP=${config.MIN_MARKET_CAP:,.0f}  MIN_PRICE=${config.MIN_PRICE}  "
      f"ATR%=[{config.MIN_ATR_PCT}, {config.MAX_ATR_PCT}]")
print(f"  Scanner  : MIN_SCORE={config.MIN_SCORE}  TOP_N={config.TOP_N}  "
      f"RSI=[{config.RSI_MIN}, {config.RSI_MAX}]")
print(f"  Gate 3   : MIN_CONFIDENCE={config.MIN_CONFIDENCE}")
print(f"  Gate 5   : MIN_EDGE_PCT={config.MIN_EDGE_PCT:.0%}  WIN_PROB_BASE={config.WIN_PROB_BASE:.0%}")
print(f"  Risk     : MAX_OPEN={config.MAX_OPEN_POSITIONS}  MAX_DAILY_LOSS={config.MAX_DAILY_LOSS_PCT:.0%}  "
      f"MAX_DRAWDOWN={config.MAX_DRAWDOWN_PCT:.0%}  KELLY={config.KELLY_FRACTION}")
print(f"  Watchlist: {WATCHLIST_PATH}")

Active strategy dials (edit in backend/config.py):
  Universe : MIN_MARKET_CAP=$100,000,000,000  MIN_PRICE=$10.0  ATR%=[1.0, 5.0]
  Scanner  : MIN_SCORE=2  TOP_N=15  RSI=[50, 70]
  Gate 3   : MIN_CONFIDENCE=6
  Gate 5   : MIN_EDGE_PCT=4%  WIN_PROB_BASE=35%
  Risk     : MAX_OPEN=5  MAX_DAILY_LOSS=3%  MAX_DRAWDOWN=8%  KELLY=0.25
  Watchlist: /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv


## Stage 1 — Universe Filter  *(weekly screen → `watchlist.csv`)*

The Tier-1 screen defines **which stocks are even allowed onto the watchlist**. It asks
TradingView for US large-caps and keeps only the liquid, tradable ones, then drops names with
earnings in the next few days. The survivors (with `price, volume, atr, rsi, sma20, sma50,
sector`) are written to `watchlist.csv`.

Dials (`config.py`): `MIN_MARKET_CAP` (≥ \$100B), `MIN_AVG_VOLUME` (≥ 1M shares/day),
`MIN_PRICE` (≥ \$10), `MIN_ATR_PCT`/`MAX_ATR_PCT` (1–5% daily range), `EARNINGS_WINDOW_DAYS`
(exclude names reporting within 5 days), `SCREENER_LIMIT`.

This is a **weekly** job and a live regeneration overwrites the existing watchlist (TradingView
+ Finnhub calls). So it is **opt-in**: by default we just read the current `watchlist.csv` and
report what's in it. Flip `RUN_UNIVERSE_FILTER = True` to rebuild it live.

In [2]:
RUN_UNIVERSE_FILTER = False   # set True to rebuild watchlist.csv live (slow; overwrites!)

if RUN_UNIVERSE_FILTER:
    # max_price passed explicitly to skip the Alpaca portfolio lookup (reproducible runs).
    count = run_universe_filter(max_price=400.0)
    print(f'[universe] regenerated watchlist with {count} stocks')

universe_df = pd.read_csv(WATCHLIST_PATH)
print(f'Universe: {len(universe_df)} stocks in watchlist.csv\n')

# Sector breakdown — how the universe is spread across the market.
print('By sector:')
print(universe_df['sector'].value_counts().to_string())

print('\nSample rows:')
universe_df.head(10)

Universe: 74 stocks in watchlist.csv

By sector:
sector
Finance                  16
Electronic Technology    11
Health Technology        10
Technology Services       9
Retail Trade              6
Consumer Non-Durables     4
Consumer Services         4
Communications            3
Energy Minerals           3
Utilities                 3
Transportation            2
Non-Energy Minerals       2
Consumer Durables         1

Sample rows:


,ticker,price,volume,atr,atr_pct,rsi,sma20,sma50,sector
0,NVDA,194.83,160196924,7.12,3.66,41.2,203.48,209.80,Electronic Technology
1,T,20.58,90520348,0.72,3.51,28.6,22.37,24.09,Communications
2,AAPL,308.63,86205596,8.74,2.83,60.3,294.80,293.52,Electronic Technology
3,AMZN,242.67,84699409,8.21,3.38,48.7,240.25,255.42,Retail Trade
4,PFE,24.32,64771998,0.58,2.37,39.8,25.13,25.71,Health Technology
5,MSFT,390.49,62903966,13.07,3.35,49.8,386.96,407.60,Technology Services
6,NFLX,77.65,58355769,2.64,3.39,47.3,77.27,84.12,Technology Services
7,TSLA,393.45,49247329,19.62,4.99,46.8,399.16,406.42,Consumer Durables
8,VZ,42.56,42174843,1.41,3.32,34.2,45.49,46.66,Communications
9,BAC,58.73,38684505,1.17,2.00,70.1,56.41,53.65,Finance


## Stage 2 — Momentum Scanner  *(nightly → top candidates)*

The Tier-2 scan scores every watchlist stock **0–3**, one point each for:

1. RSI inside the momentum zone `[RSI_MIN, RSI_MAX]` — rising but not yet overbought,
2. price **>** SMA20 — short-term uptrend,
3. SMA20 **>** SMA50 — bullish structure (golden-cross alignment).

It keeps names scoring ≥ `MIN_SCORE` and returns the strongest `TOP_N` by score.

Dials (`config.py`): `MIN_SCORE` (=2), `TOP_N` (=15), `RSI_MIN`/`RSI_MAX` (50–70).

`run_scan()` includes the `sector` column directly (Gates 1 & 4 both need `candidate['sector']`).

In [3]:
candidates_df = run_scan()   # uses MIN_SCORE / TOP_N from config

if candidates_df is None or candidates_df.empty:
    print('No candidates from run_scan — check watchlist.csv')
else:
    print(f'\nFunnel so far: {len(universe_df)} universe '
          f'→ {len(candidates_df)} candidates (score ≥ {config.MIN_SCORE}, top {config.TOP_N})')

candidates_df

[scanner] loaded 74 stocks from /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv
[scanner] 50 stocks with score >= 2
[scanner] returning top 15 candidates

Funnel so far: 74 universe → 15 candidates (score ≥ 2, top 15)


,ticker,price,score,atr,rsi,sma20,sma50,sector
0,AAPL,308.63,3,8.74,60.3,294.80,293.52,Electronic Technology
1,RTX,199.25,3,4.68,68.1,185.52,179.76,Electronic Technology
2,MO,72.71,3,1.63,54.9,71.52,71.02,Consumer Non-Durables
3,CVS,104.72,3,2.36,69.9,100.73,93.41,Retail Trade
4,FTNT,156.25,3,5.67,67.1,147.78,126.81,Technology Services
5,PM,182.27,3,5.03,54.2,179.68,177.85,Consumer Non-Durables
6,HD,357.90,3,8.44,69.5,332.69,323.30,Retail Trade
7,IBKR,91.33,3,3.89,54.0,90.66,86.07,Finance
8,BNY,146.62,3,3.36,59.6,144.06,139.19,Finance
9,APH,164.59,3,7.34,57.3,158.85,146.10,Electronic Technology


## Run-level inputs

Before the gate loop we set the **portfolio context** Gate 1 and the Risk Gate need, and fetch
the **market-wide data once** (`get_shared_market_data()` → VIX / SPY / hours-to-next-macro) so
every Gate 1 call reuses it instead of re-hitting the API.

`TOP_N` caps how many candidates we push through the (paid) Claude gates. `open_positions_count`
and `drawdown_pct` feed the Risk Gate's portfolio-limit checks.

In [4]:
portfolio_value        = 100_000.0   # total account value (Gate 1 + Risk Gate sizing)
daily_pnl              = 0.0          # today's realised + unrealised P&L (negative = loss)
open_positions_count   = 1            # currently open positions (Risk Gate limit)
drawdown_pct           = 0.01         # drawdown from peak equity, e.g. 0.01 = down 1%
TOP_N                  = 10           # cap candidates pushed through the Claude gates

shared = get_shared_market_data()   # VIX / SPY / macro — fetched once for all Gate 1 calls
print('Shared market data:', shared)
print(f'Portfolio: ${portfolio_value:,.0f}  daily_pnl=${daily_pnl:,.0f}  '
      f'open_positions={open_positions_count}  drawdown={drawdown_pct:.1%}')

Shared market data: {'vix': {'level': 16.15, 'change_pct_today': -0.0265, 'prior_close': 16.59}, 'spy': {'price': 744.78, 'change_pct_today': -0.0013, 'prior_close': 745.76}, 'macro_hours': None}
Portfolio: $100,000  daily_pnl=$0  open_positions=1  drawdown=1.0%


## Gates 1–5 & Risk — the decision chain

Each candidate runs the gates **in order, stopping at the first failure** (fail fast keeps
Claude cost down). The chain:

- **Gate 1 — Hard Threat** *(rules)*: blocks on macro/market shocks — VIX spike, SPY/sector
  selloff, pre-market gap, imminent macro event, earnings tomorrow, fresh 8-K, daily loss
  limit. Thresholds in `config.BLOCK_THRESHOLDS`.
- **Gate 2 — News Threat** *(Claude)*: reads the headlines, blocks on a catastrophic story
  (fraud, recall, regulatory action, …). News is fetched **once** here and reused by Gate 3.
- **Gate 3 — Sentiment** *(Claude)*: returns direction + confidence (0–10). `MIN_CONFIDENCE`
  turns that into pass / block / pass-with-caution.
- **Gate 4 — Contradiction** *(Claude)*: blocks on HIGH-risk contradictions; LOW/MEDIUM flags
  pause the run and prompt **Y/N** — Y continues to Gate 5, N skips the ticker.
- **Gate 5 — Edge / EV** *(rules)*: maps momentum score + Gate 3 sentiment to a win
  probability, computes expected value, and issues **BUY** if `EV ≥ MIN_EDGE_PCT` else
  **SKIP**. Also returns the trade levels (entry/stop/target).
- **Risk Gate** *(rules, zero Claude cost)*: last-mile check on every Gate 5 **BUY** — open
  position count, drawdown kill switch, daily loss limit, reward:risk floor, and Quarter-Kelly
  position sizing.

The driver below records, for every ticker, where it stopped and the Gate 3 / Gate 5 / Risk
numbers so the results table tells the whole story.

In [5]:
def _confirm_flag(ticker, g4):
    """Ask human Y/N when Gate 4 flags a contradiction for review."""
    print(f'\n[review] {ticker} flagged — {g4["contradiction_type"]} risk={g4["risk_level"]}')
    print(f'         {g4["reason"]}')
    while True:
        answer = input(f'Proceed with {ticker}? [Y/N]: ').strip().upper()
        if answer == 'Y':
            return True
        if answer == 'N':
            return False
        print('Please enter Y or N.')

def _row(ticker, decision, g3=None, g5=None, risk=None):
    """One results-table row; gate3/gate5/risk fields filled only when those gates ran."""
    tl = g5.get('trade_levels') if g5 else None
    pos = risk['position'] if risk else None
    return {
        'ticker': ticker,
        'final_decision': decision,
        'g3_direction': g3.get('direction') if g3 else None,
        'g3_confidence': g3.get('confidence') if g3 else None,
        'ev': round(g5['expected_value'], 3) if g5 else None,
        'win_prob': round(g5['win_probability'], 3) if g5 else None,
        'position_confidence': g5['position_confidence'] if g5 else None,
        'entry': tl['entry'] if tl else None,
        'stop': tl['stop'] if tl else None,
        'target': tl['target'] if tl else None,
        'reward_risk': tl['reward_risk'] if tl else None,
        'shares': pos['shares'] if pos else None,
        'position_value': pos['position_value'] if pos else None,
        'position_pct': pos['position_pct'] if pos else None,
        'risk_reject': risk.get('reject_reason') if risk and not risk['approved'] else None,
    }

rows, processed = [], 0

if candidates_df is not None and not candidates_df.empty:
    for _, r in candidates_df.iterrows():
        if processed >= TOP_N:
            break
        ticker = str(r.at['ticker'])
        sector_val = r.at['sector']
        if not isinstance(sector_val, str):
            print(f'{ticker:<5} skipped — no sector in watchlist')
            continue
        sector = sector_val
        processed += 1

        candidate = {
            'ticker': ticker,
            'sector': sector,
            'price': float(r['price']),
            'atr': float(r['atr']),
            'score': int(r['score']),
        }

        # Gate 1 — hard threats (rules, reuses shared market data)
        g1 = screen_gate1_hard_threats(candidate, shared, portfolio_value, daily_pnl)
        if not g1['passed']:
            rows.append(_row(ticker, f"BLOCKED_G1:{g1.get('block_reason')}"))
            continue

        # Gates 2 & 3 share a single news fetch
        headlines = fetch_news(ticker) or []
        g2 = assess_gate2_news_threat(candidate, headlines)
        if not g2['passed']:
            rows.append(_row(ticker, 'BLOCKED_G2'))
            continue
        g3 = evaluate_gate3_sentiment(candidate, headlines)
        if not g3['passed']:
            rows.append(_row(ticker, 'BLOCKED_G3', g3=g3))
            continue

        # Gate 4 — contradiction vs the live market backdrop
        market_context = get_market_context(sector)
        if market_context is None:
            rows.append(_row(ticker, 'BLOCKED_G4:no_market_context', g3=g3))
            continue
        g4 = detect_gate4_contradiction(candidate, g3, market_context)
        if g4['action'] == 'BLOCK':
            rows.append(_row(ticker, 'BLOCKED_G4', g3=g3))
            continue
        if g4['action'] == 'FLAG_FOR_REVIEW':
            if not _confirm_flag(ticker, g4):
                rows.append(_row(ticker, 'REJECTED_FLAG', g3=g3))
                continue

        # Gate 5 — edge check + EV
        g5 = decide_gate5_signal(candidate, {'gate1': g1, 'gate2': g2, 'gate3': g3, 'gate4': g4})
        if g5['decision'] != 'BUY':
            rows.append(_row(ticker, g5['decision'], g3=g3, g5=g5))
            continue

        # Risk gate — sizing + portfolio limits (rules only, zero Claude cost)
        risk = validate_trade(ticker, g5, portfolio_value, daily_pnl,
                              open_positions_count, drawdown_pct)
        decision = 'BUY' if risk['approved'] else f"REJECTED_RISK:{risk['reject_reason']}"
        rows.append(_row(ticker, decision, g3=g3, g5=g5, risk=risk))

print(f'\nProcessed {processed} candidates through the gates.')

[gate1] AAPL: BLOCKED — sector
[gate1] RTX: BLOCKED — sector
[gate1] MO: passed all 8 checks
[gate2] MO: passed — no threat across 5 headlines
[gate3] MO: BLOCKED — NEUTRAL conf=3: The headlines are generic dividend-income educational content with no specific mention of MO, making them irrelevant to assessing bullish momentum for this stock.
[gate1] CVS: passed all 8 checks
[gate2] CVS: passed — no threat across 1 headlines
[gate3] CVS: passed — BULLISH conf=8
[gate4] CVS: FLAG_FOR_REVIEW — divergence risk=MEDIUM: CVS is a bullish momentum candidate while its sector (XLY, -0.82%) and broad market (SPY, -0.13%) are both negative, creating relative strength divergence that risks mean reversion.

[review] CVS flagged — divergence risk=MEDIUM
         CVS is a bullish momentum candidate while its sector (XLY, -0.82%) and broad market (SPY, -0.13%) are both negative, creating relative strength divergence that risks mean reversion.
[gate5] CVS: BUY — EV 0.875 | win_prob=62% | HIGH
[risk] CVS

## Results — the funnel & the approved BUY list

The table below shows every processed ticker, where it stopped, and the Gate 3 / Gate 5 / Risk
numbers. Then the funnel counts (`universe → scanned → processed → Gate-5 BUY → approved`) and,
for each **approved BUY**, the trade levels and Quarter-Kelly position size the Risk Gate computed.

In [6]:
results_df = pd.DataFrame(rows)

if results_df.empty:
    print('No candidates were processed.')
else:
    print(results_df.to_string(index=False))

    approved = results_df[results_df['final_decision'] == 'BUY']
    gate5_buys = results_df[
        results_df['final_decision'].eq('BUY')
        | results_df['final_decision'].str.startswith('REJECTED_RISK', na=False)
    ]
    risk_rejected = results_df[results_df['final_decision'].str.startswith('REJECTED_RISK', na=False)]
    scan_df = candidates_df if candidates_df is not None else pd.DataFrame()
    print(f'\nFunnel: {len(universe_df)} universe '
          f'→ {len(scan_df)} scanned '
          f'→ {processed} processed '
          f'→ {len(gate5_buys)} Gate-5 BUY '
          f'→ {len(approved)} approved')

    if not approved.empty:
        print('\nApproved trades (Risk Gate):')
        for _, b in approved.iterrows():
            print(f"  {b['ticker']:<5} {int(b['shares'])} shares (${b['position_value']:,.2f}, "
                  f"{b['position_pct']:.1%})  "
                  f"entry={b['entry']:.2f}  stop={b['stop']:.2f}  target={b['target']:.2f}  "
                  f"R:R={b['reward_risk']:.1f}  EV={b['ev']}  conf={b['position_confidence']}")

    if not risk_rejected.empty:
        print('\nGate-5 BUY rejected by Risk Gate:')
        for _, r in risk_rejected.iterrows():
            print(f"  {r['ticker']:<5} reason={r['risk_reject']}  EV={r['ev']}")

ticker    final_decision g3_direction  g3_confidence    ev  win_prob position_confidence  entry    stop  target  reward_risk  shares  position_value  position_pct risk_reject
  AAPL BLOCKED_G1:sector         None            NaN   NaN       NaN                None    NaN     NaN     NaN          NaN     NaN             NaN           NaN        None
   RTX BLOCKED_G1:sector         None            NaN   NaN       NaN                None    NaN     NaN     NaN          NaN     NaN             NaN           NaN        None
    MO        BLOCKED_G3      NEUTRAL            3.0   NaN       NaN                None    NaN     NaN     NaN          NaN     NaN             NaN           NaN        None
   CVS               BUY      BULLISH            8.0 0.875     0.625                HIGH 104.72 101.180   111.8          2.0    76.0          8000.0        0.0800        None
  FTNT BLOCKED_G1:sector         None            NaN   NaN       NaN                None    NaN     NaN     NaN          NaN 

## Free-play

Scratch cell — tweak `portfolio_value`, `open_positions_count`, `drawdown_pct`, `TOP_N`, or a
single candidate and re-run pieces above.